In [1]:
!pip install gdown

import gdown
import pandas as pd

url = "https://drive.google.com/uc?id=19I3rLjwEXDtjeebOoTMW4vLYa3UVyklc&export=download"

output = "data.csv"
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=19I3rLjwEXDtjeebOoTMW4vLYa3UVyklc&export=download
To: /content/data.csv
100%|██████████| 2.17M/2.17M [00:00<00:00, 153MB/s]


'data.csv'

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
# Load the aug_unaccented_reviews.jsonl file into a pandas DataFrame
df = pd.read_csv('data.csv', encoding='utf-8')

# Data Inspection
print(df.shape)
print(df.columns)
print(df["label"].value_counts())
display(df.head(10000))

(31460, 4)
Index(['comment', 'label', 'rate', 'Unnamed: 3'], dtype='object')
label
POS    20093
NEG     6669
NEU     4698
Name: count, dtype: int64


,comment,label,rate,Unnamed: 3
0,Áo bao đẹp ạ!!,POS,5,NaN
1,Tuyệt vời !,POS,5,NaN
2,2day ao khong giong trong.,NEG,1,NaN
3,"Mùi thơm,bôi lên da mềm da.",POS,5,NaN
4,"Vải đẹp, dày dặn.",POS,5,NaN
...,...,...,...,...
9995,Lạp xưởng nạc nhiều vừa miệng rất ngon.,POS,5,NaN
9996,Tks shop.,POS,5,NaN
9997,Shop phục vụ kém mua sơn móng lựa màu gửi đàng...,NEG,1,NaN
9998,Shop phục vụ rất tốt.,POS,4,NaN


In [4]:
# Remove unnecessary unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [5]:
# Keep useful columns
df = df[["comment", "label", "rate"]]

In [6]:
# Remove missing values
df = df.dropna(subset=["comment", "label"])

# Remove duplicated comments
df = df.drop_duplicates(subset=["comment"])

# Standardize existing labels
df["label"] = df["label"].astype(str).str.upper().str.strip()

label_mapping = {
    "POS": "POSITIVE",
    "NEU": "NEUTRAL",
    "NEG": "NEGATIVE",
    "POSITIVE": "POSITIVE",
    "NEUTRAL": "NEUTRAL",
    "NEGATIVE": "NEGATIVE"
}

df["sentiment"] = df["label"].map(label_mapping)

# Remove rows with invalid labels
df = df.dropna(subset=["sentiment"])

In [7]:
# Vietnamese word tokenization

VIET_ABBREVIATIONS = {
    "ko":    "không", "k":     "không", "kh":    "không", "khong": "không",
    "dc":    "được",  "đc":    "được",  "dk":    "được",
    "mk":    "mình",  "mh":    "mình",
    "cx":    "cũng",  "vs":    "với",
    "r":     "rồi",   "rr":    "rồi",
    "nha":   "nhé",   "nhe":   "nhé",
    "ok":    "ổn",    "oke":   "ổn",    "okey":  "ổn",    "okie":  "ổn",
    "sp":    "sản phẩm", "sl": "số lượng", "gh": "giao hàng", "đh": "đặt hàng",
    "ad":    "shop",  "mn":    "mọi người", "mng": "mọi người",
    "iu":    "yêu",   "thik":  "thích", "thix":  "thích",
    "wa":    "quá",   "wá":    "quá",   "qá":    "quá",
    "vz":    "vậy",   "z":     "vậy",
    "tks":   "cảm ơn", "thks": "cảm ơn", "thanks": "cảm ơn",
    "ship":  "giao hàng", "shop": "cửa hàng",
    "date":  "hạn sử dụng", "2day": "hôm nay", "a": "anh", "e": "em","sz": "size","ng": "người", "bth": "bình thường", "nchung": "nói chung"
}

def normalize_abbreviations(text: str) -> str:
    words = text.split()
    return " ".join(VIET_ABBREVIATIONS.get(w.lower(), w) for w in words)

def normalize_repeated_chars(text: str) -> str:
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def remove_emoji(text: str) -> str:
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF\U0001F900-\U0001F9FF"
        "\U00002600-\U000026FF"
        "]+", flags=re.UNICODE,
    )
    return emoji_pattern.sub(" ", text)

VIET_CHARS = (
    "a-zA-Z0-9"
    "àáâãèéêìíòóôõùúýăđơư"
    "ạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỷỹỵ"
    "ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ"
    "ẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼẾỀỂỄỆỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪỬỮỰỲỶỸỴ"
    r"\s"
)

def clean_text(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return ""

    text = text.lower()

    # Remove emoji
    text = remove_emoji(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize repeated characters, for example: đẹppppp -> đẹp
    text = normalize_repeated_chars(text)

    # Remove punctuation and special characters first
    text = re.sub(f"[^" + VIET_CHARS + "]", " ", text)

    # Remove extra spaces before abbreviation normalization
    text = re.sub(r"\s+", " ", text).strip()

    # Normalize Vietnamese abbreviations after punctuation removal
    text = normalize_abbreviations(text)

    # Remove single-character tokens
    text = re.sub(r"\b[a-zA-Z0-9]\b", " ", text)

    # Remove extra spaces again
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning
df["clean_comment"] = df["comment"].apply(clean_text)

# Drop rows with missing comment
df = df[df["clean_comment"].str.strip() != '']

In [8]:
# Check cleaned dataset
print("Cleaned shape:", df.shape)

print("\nClass distribution:")
print(df["sentiment"].value_counts())

print("\nSample cleaned data:")
display(df[["clean_comment", "rate", "sentiment"]].head(1000))

Cleaned shape: (26741, 5)

Class distribution:
sentiment
POSITIVE    16183
NEGATIVE     6321
NEUTRAL      4237
Name: count, dtype: int64

Sample cleaned data:


,clean_comment,rate,sentiment
0,áo bao đẹp ạ,5,POSITIVE
1,tuyệt vời,5,POSITIVE
2,hôm nay ao không giong trong,1,NEGATIVE
3,mùi thơm bôi lên da mềm da,5,POSITIVE
4,vải đẹp dày dặn,5,POSITIVE
...,...,...,...
1062,thế này lại phải mua thường xuyên rồi,5,POSITIVE
1063,cảm ơn cửa hàng vì sự dễ thương nên có nhu cầu...,5,POSITIVE
1064,chất lượng sản phẩm kém sai màu không giống tr...,1,NEGATIVE
1065,mua nhiều được tặng quà xinh xinh,5,POSITIVE


In [9]:
print("CLEAN")
print(df['sentiment'].value_counts())

CLEAN
sentiment
POSITIVE    16183
NEGATIVE     6321
NEUTRAL      4237
Name: count, dtype: int64


In [10]:
# Save clean data to a seperate file for training

# Select only the columns you want
export_df = df[["clean_comment", "sentiment", "rate"]]

# Save to CSV
export_df.to_csv("cleaned_data.csv", index=False, encoding="utf-8-sig")

from google.colab import files

files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>